# Explore the Presidential Cognition Corpus

Starter notebook for the **analysis phase**. The corpus build (collect → clean → dedupe → classify → metrics) is handled by the scripts; here we just load `metadata.parquet` and look around. This is descriptive exploration — not cognition diagnosis.

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if (Path.cwd().name == 'notebooks') else Path.cwd()
df = pd.read_parquet(ROOT / 'data_clean' / 'metadata.parquet')
for c in ['word_count','flesch_kincaid_grade','type_token_ratio','mean_sentence_length','question_answer_ratio']:
    df[c] = pd.to_numeric(df[c], errors='coerce')
df['date'] = pd.to_datetime(df['date'], errors='coerce')
print(df.shape)
df.head()

## Coverage: transcripts per president, and over time

In [ ]:
print(df['president'].value_counts())
print()
print(df.groupby([df['date'].dt.year, 'president']).size().unstack(fill_value=0).tail(20))

## Work with canonical, in-scope transcripts only
Drop duplicate cluster members and keep reasonable-length texts.

In [ ]:
canon = df[(df['is_canonical'] == '1') & (df['word_count'] >= 200)].copy()
print('canonical, >=200 words:', len(canon))
canon.groupby('event_type')['word_count'].agg(['count','mean','median']).round(1)

## Example: readability over time, controlling for event type
Confounds matter — a rally and a written address are not comparable. Always facet by `event_type` (and ideally `prepared_or_impromptu`) before comparing presidents or years.

In [ ]:
import matplotlib.pyplot as plt

subset = canon[canon['event_type'].isin(['rally','remarks','press_conference'])].dropna(subset=['flesch_kincaid_grade'])
fig, ax = plt.subplots(figsize=(10,5))
for pres, g in subset.groupby('president'):
    g = g.sort_values('date')
    ax.scatter(g['date'], g['flesch_kincaid_grade'], s=8, alpha=0.5, label=pres)
ax.set_ylabel('Flesch-Kincaid grade'); ax.set_xlabel('date'); ax.legend(markerscale=2, fontsize=8)
ax.set_title('Readability over time (rally/remarks/press_conference only)')
plt.show()

## Next steps (analysis phase)
- Embeddings with `sentence-transformers` for semantic drift / topic-switch frequency
- Lexical diversity (MTLD/HD-D) beyond raw TTR (which is length-sensitive)
- Within-president aging models with event_type + prepared/impromptu as covariates
- Run intense LLM passes against the local LM Studio model via `scripts/llm.py`